In [1]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam

In [5]:
# Path to local dataset folder
plantvillage_dir = 'plantvillage dataset/color'

if not os.path.exists(plantvillage_dir):
    raise FileNotFoundError(f"Directory '{plantvillage_dir}' not found. Verify your folder path.")

# ==================== 1. AUTOMATIC IMAGE & CLASS DISCOVERY ====================
filepaths, labels = [], []
for root, dirs, files in os.walk(plantvillage_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            filepaths.append(os.path.join(root, file))
            labels.append(os.path.basename(root))  # Parent directory name is the class label

df_all = pd.DataFrame({'filename': filepaths, 'class': labels})

# Extract unique class names directly from discovered image locations
all_classes = sorted(df_all['class'].unique().tolist())
num_classes = len(all_classes)

print(f"Total Discovered Images : {len(df_all)}")
print(f"Total Unique Classes   : {num_classes}")

# ==================== 2. NON-IID FARM NODE PARTITIONING ====================
node_1_classes = [c for c in all_classes if c.startswith(('Apple', 'Cherry', 'Grape', 'Peach', 'Blueberry'))]
node_2_classes = [c for c in all_classes if c.startswith(('Corn', 'Pepper', 'Squash', 'Soybean', 'Orange'))]
node_3_classes = [c for c in all_classes if c.startswith(('Potato', 'Tomato', 'Strawberry', 'Raspberry'))]

df_node1 = df_all[df_all['class'].isin(node_1_classes)].reset_index(drop=True)
df_node2 = df_all[df_all['class'].isin(node_2_classes)].reset_index(drop=True)
df_node3 = df_all[df_all['class'].isin(node_3_classes)].reset_index(drop=True)

print("\n--- FARM NODE DISTRIBUTION ---")
print(f"Node 1 (Iyin-Ekiti)   - Tree Crops & Berries : {len(df_node1)} images ({len(node_1_classes)} classes)")
print(f"Node 2 (Ado-Ekiti)    - Grains & Citrus      : {len(df_node2)} images ({len(node_2_classes)} classes)")
print(f"Node 3 (Ikere-Ekiti)  - Nightshades & Herbs  : {len(df_node3)} images ({len(node_3_classes)} classes)")

Total Discovered Images : 54305
Total Unique Classes   : 38

--- FARM NODE DISTRIBUTION ---
Node 1 (Iyin-Ekiti)   - Tree Crops & Berries : 13298 images (13 classes)
Node 2 (Ado-Ekiti)    - Grains & Citrus      : 18759 images (9 classes)
Node 3 (Ikere-Ekiti)  - Nightshades & Herbs  : 22248 images (16 classes)


### Local Farm Data Generators

In [6]:
datagen = ImageDataGenerator(rescale=1./255)

def create_farm_generator(df_node, batch_size=32):
    return datagen.flow_from_dataframe(
        dataframe=df_node,
        x_col='filename',
        y_col='class',
        target_size=(128, 128),
        batch_size=batch_size,
        classes=all_classes,       # Maintains global 38-class dimension across nodes
        class_mode='categorical',
        shuffle=True
    )

node_generators = [
    create_farm_generator(df_node1),
    create_farm_generator(df_node2),
    create_farm_generator(df_node3)
]

Found 13298 validated image filenames belonging to 38 classes.
Found 18759 validated image filenames belonging to 38 classes.
Found 22248 validated image filenames belonging to 38 classes.


### CNN Model & Differential Privacy Setup

In [7]:
def build_federated_cnn(input_shape=(128, 128, 3), num_classes=38):
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False  # Freeze base layers for edge device efficiency
    
    model = Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dropout(0.2),
        Dense(128, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

NOISE_SCALE = 0.005  # Differential Privacy noise magnitude

def apply_differential_privacy(weights, noise_scale=NOISE_SCALE):
    dp_weights = []
    for w in weights:
        noise = np.random.normal(loc=0.0, scale=noise_scale, size=w.shape)
        dp_weights.append(w + noise)
    return dp_weights

### Federated Training Loop (FedAvg + Differential Privacy)

In [8]:
NUM_CLIENTS = 3
NUM_ROUNDS = 5
EPOCHS_PER_ROUND = 1

local_models = [build_federated_cnn(num_classes=num_classes) for _ in range(NUM_CLIENTS)]
global_model = build_federated_cnn(num_classes=num_classes)

print("Starting Federated Learning Loop across 3 Farm Nodes...")

for round_num in range(1, NUM_ROUNDS + 1):
    print(f"\n--- Federated Round {round_num}/{NUM_ROUNDS} (Differential Privacy Enabled) ---")
    global_weights = global_model.get_weights()
    local_weights_list = []
    round_payload_bytes = 0

    for i in range(NUM_CLIENTS):
        # Synchronize local model with global server weights
        model = local_models[i]
        model.set_weights(global_weights)
        
        # Train on local node data
        gen = node_generators[i]
        steps = min(len(gen), 30)  # Capped for faster round iteration
        model.fit(gen, steps_per_epoch=steps, epochs=EPOCHS_PER_ROUND, verbose=0)
        
        # Extract parameters and apply noise
        raw_weights = model.get_weights()
        private_weights = apply_differential_privacy(raw_weights)
        
        # Calculate bandwidth payload
        payload_bytes = sum(w.nbytes for w in private_weights)
        round_payload_bytes += payload_bytes
        
        local_weights_list.append(private_weights)
        print(f"  Farm Node {i+1} -> Local Training Complete | DP Noise Injected | Update Payload: {payload_bytes / (1024 * 1024):.2f} MB")
        
    # Aggregate weights using FedAvg
    avg_weights = [np.mean(w_tuple, axis=0) for w_tuple in zip(*local_weights_list)]
    global_model.set_weights(avg_weights)
    print(f"Total Round Payload Exchanged: {round_payload_bytes / (1024 * 1024):.2f} MB")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 38s 4us/step
Starting Federated Learning Loop across 3 Farm Nodes...

--- Federated Round 1/5 (Differential Privacy Enabled) ---
  Farm Node 1 -> Local Training Complete | DP Noise Injected | Update Payload: 18.52 MB
  Farm Node 2 -> Local Training Complete | DP Noise Injected | Update Payload: 18.52 MB
  Farm Node 3 -> Local Training Complete | DP Noise Injected | Update Payload: 18.52 MB
Total Round Payload Exchanged: 55.55 MB

--- Federated Round 2/5 (Differential Privacy Enabled) ---
  Farm Node 1 -> Local Training Complete | DP Noise Injected | Update Payload: 18.52 MB
  Farm Node 2 -> Local Training Complete | DP Noise Injected | Update Payload: 18.52 MB
  Farm Node 3 -> Local Training Complete | DP Noise Injected | Update Payload: 18.52 MB
Total Round Payload Exchanged: 55.55 MB

--- Federated Round 3/5 (Differential Privacy Enabled) ---
  Farm Node 1 -> Local Training Complete | DP Noise Injected | Update Payload: 18.52 MB
  Farm Node 2 -> L

### Save Federated Model

In [9]:
os.makedirs('models/saved_models', exist_ok=True)
global_model.save('models/saved_models/federated_global_model.h5')
print("Global Federated CNN model successfully saved to 'models/saved_models/federated_global_model.h5'.")

Global Federated CNN model successfully saved to 'models/saved_models/federated_global_model.h5'.
